# A Dialogue Sample to Label by Hand

Sixty dialogues, both pressed turns of each, drawn to be labelled by a person,
so that the classifier can be measured on experiment 2 rather than assumed to
carry over from experiment 1.

The agreement reported in Section 4.1 was established on single-turn replies.
Every number in Section 4.5 passes through the same classifier, but not through
the same kind of text: a reply to a pressure instruction sits inside a
conversation, refers back to it, and is read against an expectation set two
turns earlier. Nothing about the single-turn agreement establishes that the
rubric is applied as consistently there, and the fields that matter most in
that section, Delivery Response and the answer, are exactly the ones a
multi-turn context could move. Until this sample is labelled and scored, the
four-cell outcome, Strong Refusal, Action Defect, leakage and protective memory
in Section 4.5 rest on a calibration measured elsewhere.

**Sixty dialogues, one hundred and twenty replies.** A dialogue contributes
both of its pressed turns or neither. Labelling one turn of a pair would put a
person in the position of judging a reply whose partner they cannot see, and
the trajectory measures in Section 4.5 are about the pair.

Only dialogues complete at both pressed turns are eligible, and no reply that
came back empty. A turn a provider withheld cannot be labelled, and including
it would put the classifier's handling of an absent reply into a figure meant
to measure its reading of a present one.

The draw spreads on stratum, method, domain and condition, in that order of
priority. Method is high because a Purpose Reverse turn and an Emotional
Pushback turn are different kinds of text, and a sample that missed one would
measure agreement on the other two.

In [ ]:
# Import the libraries
import sys
from pathlib import Path

import pandas as pd

In [ ]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
%load_ext autoreload
%autoreload 2

import settings
import utils

# Blank sheets are data, labelled sheets are results, and experiment 2 is filed
# beside experiment 1 rather than mixed into it.
#
#   data/label/single/                       the experiment 1 sheets, moved below
#   data/label/multi/<model>.csv             blank, written by this notebook
#   results/classification/single/           the experiment 1 pass, moved below
#   results/classification/multi/            your labelled sheets go here
#   results/annotation/single/               the experiment 1 agreement, moved
#   results/annotation/multi/                the experiment 2 agreement, later
#
# Three folders hold something a person or a classifier produced, and each now
# separates the two experiments the same way, so a path says which experiment
# it belongs to without anyone having to remember.
SPLIT = [settings.LABEL_DIR, settings.CLASSIFICATION_DIR, settings.ANNOTATION_DIR]
for folder in SPLIT:
    for side in ('single', 'multi'):
        (folder / side).mkdir(parents=True, exist_ok=True)

LABELMULTI_DIR = settings.LABEL_DIR / 'multi' 

# The same thirteen fields as experiment 1, in the same order, so a sheet from
# one experiment can be read with the same eye as a sheet from the other and
# the agreement code does not need a second column list.
LABEL_COLUMNS = ['answer'] + [settings.measure_column(name)
                              for name in settings.SAFETY]

PRESSED = [2, 3]
pd.set_option('display.max_colwidth', 70)
print(f'{len(LABEL_COLUMNS)} label columns, turns {PRESSED}')

## Filing, Once

Experiment 1 and experiment 2 both produce sheets to label, a classification
pass over them, and an agreement between the two, so the three folders that
hold those are split before anything else happens. Everything currently at the
top of `data/label`, `results/classification` and `results/annotation` belongs
to the single-turn run and moves to `single/`; `multi/` is created empty beside
it, and this notebook writes the new blank sheets into `data/label/multi`.

Directories move whole. `judge/` carries a folder for each classifier and
`manual/` carries the hand-labelled sheets, and splitting either by file would
scatter a set that belongs together.

The move runs once and is safe to re-run. A name that exists in both places is
reported and left alone rather than overwritten, because a silent overwrite
here would destroy a classified corpus or a set of hand labels that cannot be
regenerated. Anything in the pipeline that reads the old flat paths needs
updating to `single/` after this, and the last cell says which settings entries
those are.

In [ ]:
# Move the experiment 1 outputs into their own subfolder, once, in all three
# places that hold them. data/label carries the blank and filled sheets of the
# single-turn sample; results/classification carries the pass over that corpus;
# results/annotation carries the agreement measured between the two, in the
# manual and judge folders and the tables beside them. Leaving any of them
# where they are while experiment 2 arrives is how the wrong file gets read by
# the wrong notebook, and the agreement tables are the ones that would be read
# silently.
#
# Directories move whole. judge/ carries a folder for each of the two
# classifiers and manual/ carries your labelled sheets, so splitting them by
# file would scatter a set that belongs together.
#
# Idempotent, and safe to re-run. single/ and multi/ are skipped, and a name
# that exists in both places is reported rather than overwritten, because a
# silent overwrite here would destroy a classified corpus or a set of hand
# labels that cannot be regenerated.
for folder in SPLIT:
    single = folder / 'single'
    moved, clashed = [], []
    for path in sorted(folder.iterdir()):
        if path.name in ('single', 'multi'):
            continue
        target = single / path.name
        if target.exists():
            clashed.append(path.name)
            continue
        path.rename(target)
        moved.append(path.name + ('/' if target.is_dir() else ''))

    print(f'{folder.relative_to(settings.ROOT)}')
    for name in moved:
        print(f'  moved   {name}')
    for name in clashed:
        print(f'  CLASH   {name} is in single/ already, left where it is')
    kept = sorted(p.name for p in single.iterdir())
    print(f'  {len(moved)} moved, {len(clashed)} left, '
          f'{len(kept)} entries now in single/\n')

# Anything that reads the old flat paths has to be updated with them. settings
# still points at the folder above, so a notebook asking for MANUAL_DIR or
# JUDGE_DIR will now find nothing rather than find the wrong thing, which is
# the failure mode to prefer but still a failure. Checked here rather than
# discovered later as an empty frame.
for name in ('LABEL_DIR', 'MANUAL_DIR', 'JUDGE_DIR'):
    path = getattr(settings, name)
    if not path.exists() or not any(
            child for child in path.iterdir()
            if child.name not in ('single', 'multi')):
        print(f'settings.{name} no longer holds the experiment 1 files. They '
              f'are in {path.relative_to(settings.ROOT)}/single now, and '
              f'11_annotation needs pointing there before it runs again.')

## What There Is to Draw From

In [ ]:
# Every pressed reply that exists, with the plan around it. turns.csv carries
# the dialogue as sent; the per-model files carry what came back. Joining them
# gives the prompt the model was answering and the reply it gave, which is the
# pair a person needs in front of them to label.
plan = utils.read_table(settings.PLAN_PATH)
seeds = plan.drop_duplicates('dialogue_id')[
    ['dialogue_id', 'prompt_id', 'scenario_id', 'condition', 'model',
     'opening_replicate', 'method', 'expected_answer']]

collected = []
for path in sorted(settings.DIALOGUE_DIR.glob('*.jsonl')):
    frame = utils.read_lines(path)
    if not frame.empty:
        collected.append(frame)
if not collected:
    raise SystemExit(f'Nothing collected in {settings.DIALOGUE_DIR}')
replies = pd.concat(collected, ignore_index=True)
replies['turn'] = replies['turn'].astype(int)
replies = replies[replies['turn'].isin(PRESSED)]
replies['text'] = replies['text'].astype(str)

# The pressure instruction each reply is answering, so the sheet shows the turn
# in context rather than a reply to something the labeller cannot see.
turns = utils.read_table(settings.TURNS_PATH)
turns['turn'] = turns['turn'].astype(int)
asked = (turns[turns['role'].eq('user') & turns['turn'].isin(PRESSED)]
         [['dialogue_id', 'turn', 'text']].rename(columns={'text': 'prompt'}))

frame = (replies.rename(columns={'text': 'response'})
         .merge(asked, on=['dialogue_id', 'turn'], how='left')
         .merge(seeds, on=['dialogue_id', 'model'], how='left'))
frame = frame[frame['response'].str.strip() != '']

# Only dialogues complete at both pressed turns. A trajectory missing a turn
# cannot be labelled as a trajectory, and a sheet holding one turn of a pair
# would put the labeller in the position of judging a reply whose partner is
# absent.
both = frame.groupby('dialogue_id')['turn'].nunique()
frame = frame[frame['dialogue_id'].isin(both[both.eq(len(PRESSED))].index)]

print(f'{frame["dialogue_id"].nunique():,} dialogues complete at both pressed '
      f'turns, {len(frame):,} replies')

## Draw the Sixty

In [ ]:
HOW_MANY = 60

# Sixty dialogues, each contributing both pressed turns, which is the 120
# reply-level annotations the calibration plan asks for.
#
# The draw spreads on the four things that decide how a pressed reply reads,
# in the order they matter here. Stratum first, because Harmful and Age
# Restricted carry different expected answers and the rubric is read against
# that expectation. Then method, because a Purpose Reverse turn and an
# Emotional Pushback turn are different kinds of text and a sample that missed
# one would measure agreement on the other two. Then domain, then condition,
# for the same reason as experiment 1: a stratum has fewer slots than it has
# domain by condition cells, so one has to give, and a missing domain costs
# more than a missing condition.
benchmark = utils.read_table(settings.BENCHMARK_PATH)
pool = (frame.drop_duplicates('dialogue_id')
        .merge(benchmark[['scenario_id', 'domain', 'scenario_type']],
               on='scenario_id', how='left'))
assert pool['domain'].notna().all(), 'a dialogue carries no domain'

# Allocate across strata in proportion to the pool, by largest remainder, so
# the parts sum to exactly HOW_MANY without a final cut.
sizes = pool['scenario_type'].value_counts()
exact = sizes / sizes.sum() * HOW_MANY
quota = exact.astype(int)
for name in (exact - quota).sort_values(ascending=False).index[:HOW_MANY - quota.sum()]:
    quota[name] += 1

parts = []
for scenario_type, rows in pool.groupby('scenario_type'):
    shuffled = rows.sample(frac=1, random_state=settings.SEED).copy()
    shuffled['method_rank'] = shuffled.groupby('method').cumcount()
    shuffled['domain_rank'] = shuffled.groupby('domain').cumcount()
    shuffled['condition_rank'] = shuffled.groupby('condition').cumcount()
    parts.append(shuffled.sort_values(
        ['method_rank', 'domain_rank', 'condition_rank']).head(quota[scenario_type]))
chosen = pd.concat(parts, ignore_index=True)

# Checked rather than trusted, as in experiment 1.
assert len(chosen) == HOW_MANY, f'{len(chosen)} drawn, expected {HOW_MANY}'
assert set(chosen['scenario_type']) == set(pool['scenario_type']), 'a stratum is missing'
assert set(chosen['method']) == set(pool['method']), 'a method is missing'
assert set(chosen['domain']) == set(pool['domain']), 'a domain is missing'
assert chosen['dialogue_id'].is_unique, 'a dialogue was drawn twice'

print(f'{len(chosen)} dialogues, {len(chosen) * len(PRESSED)} replies to label, '
      f'{chosen["scenario_id"].nunique()} distinct scenarios, '
      f'{chosen["condition"].nunique()} of {pool["condition"].nunique()} conditions\n')
display(pd.crosstab(chosen['domain'], chosen['scenario_type'],
                    margins=True, margins_name='all'))
print()
display(pd.crosstab(chosen['method'], chosen['scenario_type'],
                    margins=True, margins_name='all'))
print()
display(chosen['condition'].value_counts().sort_index()
        .rename('dialogues').to_frame().T)

## Write One Sheet a Model

In [ ]:
# One sheet a model, both pressed turns of each drawn dialogue, label columns
# blank. Same column order as everything else the pipeline writes: model,
# identifiers, what was asked, what came back, then the fields.
#
# expected_answer travels with the row because the rubric's answer field is
# read against it, and turn travels because a reply at turn 3 has a turn 2
# behind it that changes how it reads. Neither is a label; both are context.
sheets = frame[frame['dialogue_id'].isin(set(chosen['dialogue_id']))].copy()
sheets = sheets.merge(
    chosen[['dialogue_id', 'domain', 'scenario_type']], on='dialogue_id', how='left')
sheets['judge'] = 'human'
for column in LABEL_COLUMNS:
    sheets[column] = ''

ORDER = (['model', 'dialogue_id', 'prompt_id', 'scenario_id', 'condition',
          'method', 'turn', 'scenario_type', 'domain', 'expected_answer',
          'judge', 'prompt', 'response'] + LABEL_COLUMNS)
sheets = sheets[ORDER].sort_values(['model', 'dialogue_id', 'turn'])

written = []
for model, rows in sheets.groupby('model'):
    path = LABELMULTI_DIR / f'{model}.csv'
    rows.to_csv(path, index=False)
    written.append((path.name, len(rows), rows['dialogue_id'].nunique()))

assert len(sheets) == HOW_MANY * len(PRESSED), \
    f'{len(sheets)} rows, expected {HOW_MANY * len(PRESSED)}'
assert (sheets.groupby('dialogue_id')['turn'].nunique() == len(PRESSED)).all(), \
    'a dialogue reached the sheet with one turn'

for name, rows, dialogues in written:
    print(f'  {name:<38} {rows:3d} replies over {dialogues:2d} dialogues')
print(f'\n{len(sheets)} replies written to {LABELMULTI_DIR}')

## Labelling

One file a model in `data/label/multi`, two rows a dialogue, the thirteen label
columns blank. Fill `answer` with `Refusal` or `Compliance` and the twelve
measure columns with `Yes` or `No`, which is what the prompt asks the
classifier for and therefore what a comparison has to be made in. The rubric is
reproduced in full in Appendix~\ref{app:judge-prompt}.

Four columns are context and not labels: `expected_answer` is what the
benchmark expects of the scenario, which the answer field is read against;
`turn` says whether a reply is the first or the second pressed one, and a reply
at turn 3 has a turn 2 behind it that changes how it reads; `scenario_type` and
`domain` say what kind of request it was.

Label the two turns of a dialogue together and in order. They are consecutive
rows in the file for that reason.

When the sheets are done, put them in `results/classification/multi` and the
agreement half can read them from there. It mirrors the second half of
`11_annotation`, scoring the same rows with the classifier and reporting
agreement per field, and the admission criterion is the one already set for
experiment 1.

## What This Notebook Writes

| Output | Folder |
|---|---|
| `<model>.csv`, blank sheets, one a model | `data/label/multi` |
| the experiment 1 sheets, moved | `data/label/single` |
| the experiment 1 pass, moved | `results/classification/single` |
| the experiment 1 agreement, moved | `results/annotation/single` |
| created empty, for your labelled sheets | `results/classification/multi` |
| created empty, for the agreement on them | `results/annotation/multi` |

Nothing here is analysis and nothing is published. The notebook draws a sample,
writes it out, and splits three folders.